In [6]:
# ============================================================
# Data Science Term Project - Data Preprocessing
# Dataset: healthcare-dataset-stroke-data.csv
# Purpose: Handle missing values, encode categorical variables,
#          split data, and apply feature scaling.
#          Save results to processed_data.pkl for use in modeling.
# ============================================================

# =====================
# Step 1. Import Libraries
# =====================
# numpy  : numerical computation library (array operations, math functions)
# pandas : data manipulation library (DataFrame, CSV reading/writing)
# matplotlib.pyplot : 2D plotting library for visualizations
# seaborn : statistical visualization library built on top of matplotlib
# warnings : suppress non-critical warning messages during execution
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("Step 1. Libraries imported successfully")
print("=" * 60)


# =====================
# Step 2. Load Dataset
# =====================
# pd.read_csv(filepath) : reads a CSV file and returns a DataFrame
#   - filepath : path to the CSV file (string)
# dataset.shape  : returns (num_rows, num_columns) tuple
# dataset.head() : returns the first 5 rows of the DataFrame
# dataset.dtypes : returns the data type of each column
# dataset.describe() : returns summary statistics (mean, std, min, max, etc.)
#                      for all numeric columns
FILE_PATH = r"../data/healthcare-dataset-stroke-data.csv"

dataset = pd.read_csv(FILE_PATH)

print("\n" + "=" * 60)
print("Step 2. Dataset Information")
print("=" * 60)

print("\n[Dataset Size]")
print(f"  Rows    : {dataset.shape[0]}")
print(f"  Columns : {dataset.shape[1]}")

print("\n[Column List]")
print(dataset.columns.tolist())

print("\n[First 5 Rows]")
print(dataset.head())

print("\n[Data Types]")
print(dataset.dtypes)

print("\n[Basic Statistics]")
print(dataset.describe())


# =====================
# Step 3. Handle Missing Values
# =====================
# pd.to_numeric(series, errors='coerce') :
#   Converts a Series to numeric type.
#   errors='coerce' : any value that cannot be converted (e.g., "N/A" string)
#                     is replaced with NaN instead of raising an error.
#   Used here because the bmi column contains "N/A" strings instead of NaN.
#
# dataset.isnull().sum() :
#   isnull() returns a boolean DataFrame (True where value is NaN)
#   .sum() counts the number of True values per column
#
# series.mean() : calculates the arithmetic mean, ignoring NaN values
#
# series.fillna(value, inplace=True) :
#   Fills all NaN values in the Series with the specified value.
#   inplace=True : modifies the original DataFrame directly (no copy created)
#   Strategy: mean imputation chosen over row deletion to preserve data size.
#
# dataset.reset_index(drop=True) :
#   Resets the row index to 0, 1, 2, ... after filtering rows.
#   drop=True : discards the old index instead of adding it as a column.
print("\n" + "=" * 60)
print("Step 3. Handling Missing Values")
print("=" * 60)

# Convert "N/A" strings in bmi column to actual NaN
dataset['bmi'] = pd.to_numeric(dataset['bmi'], errors='coerce')

print("\n[Missing Value Count]")
print(dataset.isnull().sum())

# Replace NaN in bmi with the column mean (mean imputation)
bmi_mean = dataset['bmi'].mean()
print(f"\nbmi mean value: {bmi_mean:.2f}")

dataset['bmi'].fillna(bmi_mean, inplace=True)

print("\n[Missing Value Count After Imputation]")
print(dataset.isnull().sum())

# Remove rows where gender is 'Other' (only 1 record, would add noise)
dataset = dataset[dataset['gender'] != 'Other']
dataset = dataset.reset_index(drop=True)

print(f"\nDataset size after removing 'Other' gender: {dataset.shape}")


# =====================
# Step 4. Categorical Variable Encoding
# =====================
# Machine learning models require numeric input.
# Categorical (text) columns must be converted to numbers.
#
# Two encoding strategies are used based on number of categories:
#   - Label Encoding  : for binary categories (2 unique values)
#   - One-Hot Encoding: for multi-class categories (3+ unique values)

print("\n" + "=" * 60)
print("Step 4. Categorical Variable Encoding")
print("=" * 60)

# ------------------------------------------
# 4-1. Label Encoding (binary categorical columns)
# ------------------------------------------
# LabelEncoder (sklearn.preprocessing) :
#   Converts string labels to integer codes (0, 1, 2, ...).
#   Used ONLY for binary categories (2 values) to avoid creating
#   unintended ordinal relationships (e.g., France=0 < Germany=1 < Spain=2).
#
# label_encoder.fit_transform(series) :
#   fit()      : learns the mapping from string labels to integers
#   transform(): applies the learned mapping to convert strings to integers
#   fit_transform() does both in a single step.
#   Returns a numpy array of integer-encoded values.
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# gender: Male / Female -> 0 / 1
# Binary category -> Label Encoding is appropriate
dataset['gender_encoded'] = label_encoder.fit_transform(dataset['gender'])
print("\n[Label Encoding: gender]")
print(dataset[['gender', 'gender_encoded']].drop_duplicates())

# ever_married: No / Yes -> 0 / 1
dataset['ever_married_encoded'] = label_encoder.fit_transform(dataset['ever_married'])
print("\n[Label Encoding: ever_married]")
print(dataset[['ever_married', 'ever_married_encoded']].drop_duplicates())

# Residence_type: Rural / Urban -> 0 / 1
dataset['Residence_type_encoded'] = label_encoder.fit_transform(dataset['Residence_type'])
print("\n[Label Encoding: Residence_type]")
print(dataset[['Residence_type', 'Residence_type_encoded']].drop_duplicates())

# ------------------------------------------
# 4-2. One-Hot Encoding (multi-class categorical columns)
# ------------------------------------------
# pd.get_dummies(series, prefix='name') :
#   Creates one binary column per category value (dummy variables).
#   Each column contains 0 or 1 indicating absence or presence.
#   prefix : string to prepend to the new column names (e.g., 'work_Private')
#   Used for 3+ categories to avoid imposing false ordinal relationships.
#   Number of new columns = number of unique categories.
#
# work_type has 5 categories  -> creates 5 dummy columns
# smoking_status has 4 categories -> creates 4 dummy columns

work_type_dummies = pd.get_dummies(dataset['work_type'], prefix='work')
smoking_dummies   = pd.get_dummies(dataset['smoking_status'], prefix='smoke')

print("\n[One-Hot Encoding: work_type]")
print(work_type_dummies.head())

print("\n[One-Hot Encoding: smoking_status]")
print(smoking_dummies.head())

# pd.concat([df1, df2, ...], axis=1) :
#   Concatenates DataFrames along columns (axis=1) or rows (axis=0).
#   Here we attach the dummy variable columns to the original dataset.
dataset = pd.concat([dataset, work_type_dummies, smoking_dummies], axis=1)

# DataFrame.drop(columns=[...], inplace=True) :
#   Removes specified columns from the DataFrame.
#   The original categorical columns are removed after creating dummy variables
#   to avoid redundancy.
dataset.drop(
    columns=['gender', 'ever_married', 'Residence_type', 'work_type', 'smoking_status'],
    inplace=True
)

# Remove id column - it is a unique identifier with no predictive value.
# Keeping it could cause the model to learn meaningless patterns.
dataset.drop(columns=['id'], inplace=True)

print("\n[Column List After Encoding]")
print(dataset.columns.tolist())
print(f"\nFinal dataset size: {dataset.shape}")


# =====================
# Step 5. Train / Test Split
# =====================
# train_test_split (sklearn.model_selection) :
#   Splits arrays or DataFrames into random train and test subsets.
#   Parameters:
#     X, y         : features and target arrays to split
#     test_size    : proportion of data for test set (0.2 = 20%)
#     random_state : seed for reproducibility (same split every run)
#     stratify     : ensures the class ratio in y is preserved in both
#                    train and test sets. Critical for imbalanced data
#                    (stroke rate ~4.9%) to prevent all stroke cases
#                    ending up in one split.
print("\n" + "=" * 60)
print("Step 5. Train / Test Split")
print("=" * 60)

from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = dataset.drop(columns=['stroke'])
y = dataset['stroke']

print(f"\nFeature (X) size: {X.shape}")
print(f"Target  (y) size: {y.shape}")
print(f"\nTarget distribution (stroke rate):\n{y.value_counts()}")

# Split: 80% training, 20% testing
# stratify=y preserves class ratio in both splits (important for imbalanced data)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nX_train size: {X_train.shape}")
print(f"X_test  size: {X_test.shape}")


# =====================
# Step 6. Feature Scaling
# =====================
# StandardScaler (sklearn.preprocessing) :
#   Standardizes features by removing the mean and scaling to unit variance.
#   Formula: z = (x - mean) / std  ->  result has mean=0, std=1
#   Applied only to numeric columns (age, avg_glucose_level, bmi) because:
#     - These columns have very different ranges (age: 0-82, glucose: 55-271)
#     - Distance-based and gradient-based algorithms are sensitive to scale
#     - Binary columns (0/1) are already on the same scale, no need to scale
#
# scaler.fit_transform(X_train) :
#   fit()      : computes mean and std from X_train
#   transform(): applies the standardization using computed mean and std
#   ONLY applied to training data to prevent data leakage.
#
# scaler.transform(X_test) :
#   Applies the SAME mean and std learned from X_train to X_test.
#   DO NOT use fit_transform on test data - that would use test statistics
#   and cause data leakage (the model would indirectly "see" test data).
print("\n" + "=" * 60)
print("Step 6. Feature Scaling")
print("=" * 60)

from sklearn.preprocessing import StandardScaler

NUMERIC_COLS = ['age', 'avg_glucose_level', 'bmi']

scaler = StandardScaler()

# Apply fit_transform to training data: learn mean/std AND transform
X_train[NUMERIC_COLS] = scaler.fit_transform(X_train[NUMERIC_COLS])

# Apply transform only to test data: use training mean/std to transform
# (prevents data leakage from test set into training statistics)
X_test[NUMERIC_COLS] = scaler.transform(X_test[NUMERIC_COLS])

print("\n[X_train numeric statistics after scaling]")
print(X_train[NUMERIC_COLS].describe().round(4))

print("\n[Before vs After scaling (age column)]")
print(f"  Mean after scaling      : {X_train['age'].mean():.4f}  (should be ~0)")
print(f"  Std deviation after scaling: {X_train['age'].std():.4f}  (should be ~1)")


# =====================
# Visualization
# =====================
# matplotlib.pyplot.subplots(nrows, ncols, figsize) :
#   Creates a figure with a grid of subplots.
#   nrows, ncols : number of rows and columns of subplots
#   figsize : (width, height) in inches
#   Returns (fig, axes) where axes is a 2D array of Axes objects.
#
# fig.suptitle(text, fontsize) : sets the main title for the entire figure
#
# axes[r,c].bar(x, height, color) : draws a bar chart on subplot [r,c]
# axes[r,c].hist(data, bins, color, alpha) : draws a histogram
#   bins  : number of equally spaced bins
#   alpha : transparency (0=invisible, 1=opaque)
#
# plt.tight_layout() : automatically adjusts subplot spacing to prevent overlap
# plt.savefig(path, dpi, bbox_inches) :
#   Saves the figure to a file.
#   dpi         : resolution in dots per inch (150 = high quality)
#   bbox_inches : 'tight' crops the figure to remove extra whitespace
# plt.close() : closes the figure and frees memory
print("\n" + "=" * 60)
print("Saving preprocessing visualization...")
print("=" * 60)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Stroke Dataset - Preprocessing Result Visualization', fontsize=16, fontweight='bold')

# 1. Target class distribution
axes[0, 0].bar(['No Stroke (0)', 'Stroke (1)'],
               y.value_counts().sort_index(),
               color=['steelblue', 'tomato'])
axes[0, 0].set_title('Target Distribution (stroke)')
axes[0, 0].set_ylabel('Count')

# 2. age before scaling
axes[0, 1].hist(X['age'], bins=30, color='steelblue', alpha=0.7, label='Before')
axes[0, 1].set_title('age - Before Scaling')
axes[0, 1].set_xlabel('age')

# 3. age after scaling
axes[0, 2].hist(X_train['age'], bins=30, color='tomato', alpha=0.7, label='After')
axes[0, 2].set_title('age - After Scaling (StandardScaler)')
axes[0, 2].set_xlabel('scaled age')

# 4. avg_glucose_level before scaling
axes[1, 0].hist(X['avg_glucose_level'], bins=30, color='steelblue', alpha=0.7)
axes[1, 0].set_title('avg_glucose_level - Before Scaling')
axes[1, 0].set_xlabel('avg_glucose_level')

# 5. avg_glucose_level after scaling
axes[1, 1].hist(X_train['avg_glucose_level'], bins=30, color='tomato', alpha=0.7)
axes[1, 1].set_title('avg_glucose_level - After Scaling')
axes[1, 1].set_xlabel('scaled avg_glucose_level')

# 6. work_type one-hot encoding result
work_cols   = [c for c in dataset.columns if c.startswith('work_')]
work_counts = dataset[work_cols].sum().sort_values(ascending=False)
axes[1, 2].bar(work_counts.index, work_counts.values, color='mediumseagreen')
axes[1, 2].set_title('work_type One-Hot Encoding Result')
axes[1, 2].set_xticklabels(work_counts.index, rotation=30, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig(r'../outputs/preprocessing_result.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: preprocessing_result.png")


# =====================
# Summary
# =====================
print("\n" + "=" * 60)
print("Preprocessing Complete - Summary")
print("=" * 60)
print(f"  Total records         : {dataset.shape[0]}")
print(f"  Final feature count   : {X.shape[1]}")
print(f"  Training set size     : {X_train.shape}")
print(f"  Test set size         : {X_test.shape}")
print(f"\n  [Missing Value Treatment]")
print(f"    - bmi: replaced NaN with mean ({bmi_mean:.2f})")
print(f"\n  [Label Encoding applied to]")
print(f"    - gender, ever_married, Residence_type")
print(f"\n  [One-Hot Encoding applied to]")
print(f"    - work_type ({len(work_type_dummies.columns)} dummy columns)")
print(f"    - smoking_status ({len(smoking_dummies.columns)} dummy columns)")
print(f"\n  [Feature Scaling applied to]")
print(f"    - age, avg_glucose_level, bmi (StandardScaler)")


Step 1. Libraries imported successfully

Step 2. Dataset Information

[Dataset Size]
  Rows    : 5110
  Columns : 12

[Column List]
['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']

[First 5 Rows]
      id  gender   age  hypertension  heart_disease ever_married  \
0   9046    Male  67.0             0              1          Yes   
1  51676  Female  61.0             0              0          Yes   
2  31112    Male  80.0             0              1          Yes   
3  60182  Female  49.0             0              0          Yes   
4   1665  Female  79.0             1              0          Yes   

       work_type Residence_type  avg_glucose_level   bmi   smoking_status  \
0        Private          Urban             228.69  36.6  formerly smoked   
1  Self-employed          Rural             202.21   NaN     never smoked   
2        Private          Rural             105.92  

In [7]:
# =====================
# Step 7. Save Preprocessed Data
# =====================
# pickle (Python standard library) :
#   Serializes Python objects to binary format for saving to disk.
#   Used to save preprocessed data so modeling.ipynb can load it
#   directly without repeating the preprocessing steps.
#
# pickle.dump(obj, file) :
#   Serializes obj and writes it to the open file handle.
#   obj  : any Python object (dict, DataFrame, model, etc.)
#   file : file handle opened in binary-write mode ('wb')
#
# open(path, 'wb') :
#   Opens a file for writing in binary mode ('w'=write, 'b'=binary).
#   Required for pickle since it writes raw bytes, not text.
#
# Note: X_train and X_test are already scaled (modified in-place in Step 6),
#       so they are saved directly as 'X_train_scaled' and 'X_test_scaled'.
import pickle

save_dict = {
    'X_train_scaled': X_train,   # scaled training features (already transformed)
    'X_test_scaled' : X_test,    # scaled test features (already transformed)
    'y_train'       : y_train,   # training labels
    'y_test'        : y_test,    # test labels
    'X'             : X,         # full feature set (used for column name reference)
    'dataset'       : dataset,   # fully preprocessed dataset (used for K-Means)
    'scaler'        : scaler,    # fitted StandardScaler object (reuse for new data)
}

with open(r'../outputs/processed_data.pkl', 'wb') as f:
    pickle.dump(save_dict, f)

print("processed_data.pkl saved successfully!")


processed_data.pkl saved successfully!
